# **Penting**

*   Pada Notebook ini, Anda hanya perlu mengerjakan code pada bagian **logic.py** saja. Anda tidak diwajibkan untuk mengubah atau menambahkan **app.py** yang digunakan untuk membangun interface Streamlit.
*   Namun, jika Anda memiliki preferensi lain atau ingin mengubah struktur code pada logic ataupun pada interface Streamlit, itu **DIPERSILAHKAN** saja, tetapi pastikan untuk memenuhi kriteria yang telah ditetapkan pada intruksi submission
*   Jika Anda tidak ingin mengubah apapun dan ingin mengikuti template, tugas Anda hanyalah melengkapi code yang rumpang pada bagian yang sudah ditandai "________" saja.



# **Prepare Dependencies**

In [1]:
import sys
import subprocess

# Bootstrap pip if the kernel environment does not include it yet.
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyngrok",
    "streamlit",
    "torch",
    "diffusers",
    "transformers",
    "streamlit-drawable-canvas==0.8.0"
 ])

0

In [2]:
from pyngrok import ngrok
import subprocess

# **Streamlit**

## logic.py (**Basic**)

In [ ]:
%%writefile logic.py
import torch
import gc
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler
)
from PIL import Image, ImageFilter

# MODEL LOADER (JANGAN DIUBAH)
def load_models_cached():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading models to {device}")

    # CPU membutuhkan float32 agar proses stabil; float16 untuk GPU.
    dtype = torch.float16 if device == "cuda" else torch.float32

    pipe_txt2img = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", torch_dtype=dtype
    ).to(device)

    # Default: jangan load model inpainting agar startup localhost jauh lebih cepat.
    pipe_inpaint = None

    return pipe_txt2img, pipe_inpaint

# Ini mencegah error "Function not found" jika hanya mengerjakan Basic
def flush_memory(): pass
def set_scheduler(pipe, name): return pipe
def run_inpainting(pipe, img, mask, prompt, strength): return None
def prepare_outpainting(img, expand=128): return img, None

Overwriting logic.py


In [4]:
%%writefile -a logic.py


def generate_image(pipe, prompt, neg_prompt, seed, steps, cfg, num_images=1, scheduler_name="Euler A"):

    ### MULAI CODE ###

    # Setup Generator (Seed)
    generator = torch.Generator(device=pipe.device).manual_seed(int(seed))

    # Generate gambar standard
    image = pipe(
        prompt=prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
        generator=generator
    ).images[0]

    ### SELESAI CODE ###

    # Kembalikan dalam bentuk List agar kompatibel dengan UI (List isi 1 gambar)
    return [image]

Appending to logic.py


## logic.py (**Skilled**)

In [5]:
%%writefile -a logic.py

# Implementasi pembersihan RAM GPU
def flush_memory():

    ### MULAI CODE ###

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ### SELESAI CODE ###

    print("Memory Flushed!")

# Ganti scheduler sesuai input
def set_scheduler(pipe, scheduler_name):

    ### MULAI CODE ###

    if scheduler_name == "Euler A":
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "DPM++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "DDIM":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    ### SELESAI CODE ###

    return pipe

# Definisikan ulang fungsi generate_image dan tambahkan parameter untuk batch inference
def generate_image(pipe, prompt, neg_prompt, seed, steps, cfg, num_images=1, scheduler_name="Euler A"):

    ### MULAI CODE ###

    # Set Scheduler
    pipe = set_scheduler(pipe, scheduler_name)

    generator = [
        torch.Generator(device=pipe.device).manual_seed(int(seed) + i)
        for i in range(int(num_images))
    ]

    # Generate Batch
    result = pipe(
        prompt=[prompt] * int(num_images),
        negative_prompt=[neg_prompt] * int(num_images),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
        generator=generator
    ).images

    ### SELESAI CODE ###

    return result

Appending to logic.py


## logic.py (**Advanced**)

In [6]:
%%writefile -a logic.py

def run_inpainting(pipe, image, mask, prompt, strength):
    # Pastikan konversi RGB/L dan Resize Mask (Nearest)
    if image.mode != "RGB": image = image.convert("RGB")
    if mask.mode != "L": mask = mask.convert("L")

    # Resize Mask agar tajam
    if image.size != mask.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    result = pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        strength=float(strength)
    ).images[0]

    return result

def prepare_outpainting(image, expand_pixels=128):

    ### MULAI CODE ###
    w, h = image.size
    new_w = w + (expand_pixels * 2)
    new_h = h + (expand_pixels * 2)

    # Safety: Resolusi kelipatan 8
    new_w -= (new_w % 8)
    new_h -= (new_h % 8)

    ### SELESAI CODE ###

    # Background Blur
    bg = image.resize((new_w, new_h), resample=Image.BICUBIC)
    bg = bg.filter(ImageFilter.GaussianBlur(radius=50))

    canvas = bg.copy()
    paste_x = (new_w - w) // 2
    paste_y = (new_h - h) // 2
    canvas.paste(image, (paste_x, paste_y))

    # Masker (Putih = Edit, Hitam = Keep)
    mask = Image.new("L", (new_w, new_h), 255)
    inner_box = Image.new("L", (w, h), 0)

    ### MULAI CODE ###

    mask.paste(inner_box, (paste_x, paste_y))

    ### SELESAI CODE ###

    return canvas, mask

Appending to logic.py


## app.py
Anda **TIDAK perlu mengubah atau menambahkan** apapun pada file **app.py** ini, cukup **jalankan** saja.

In [7]:
%%writefile app.py
import streamlit as st
import logic

st.set_page_config(page_title="Image Generator", layout="centered", page_icon="🖼️")
st.title("Image Generation with Stable Diffusion")

@st.cache_resource
def get_models():
    return logic.load_models_cached()

try:
    pipe_txt2img, _ = get_models()
except Exception as e:
    st.error(f"Error loading models: {e}")
    st.stop()

prompt = st.text_input(
    "Prompt",
    value="a futuristic city at sunset, ultra detailed, cinematic lighting"
)
negative_prompt = st.text_input(
    "Negative Prompt",
    value="blurry, low quality, distorted, bad anatomy"
)

guidance_scale = st.slider("guidance_scale", min_value=1.0, max_value=20.0, value=7.5, step=0.5)
num_inference_steps = st.slider("num_inference_steps", min_value=10, max_value=60, value=30, step=1)

if st.button("Generate", type="primary"):
    if not prompt.strip():
        st.warning("Prompt tidak boleh kosong.")
    else:
        with st.spinner("Generating image..."):
            logic.flush_memory()
            generated_images = logic.generate_image(
                pipe=pipe_txt2img,
                prompt=prompt,
                neg_prompt=negative_prompt,
                seed=42,
                steps=num_inference_steps,
                cfg=guidance_scale,
                num_images=1,
                scheduler_name="Euler A"
            )

        if generated_images:
            st.image(generated_images[0], caption="Generated Image", use_container_width=True)
        else:
            st.error("Gagal menghasilkan gambar.")

Overwriting app.py


# **Menggunakan *Ngrok* Untuk Deployment**

## **Konfigurasi Autentikasi Ngrok dan Menjalankan Aplikasi Streamlit**
Cell ini digunakan untuk mengatur *authentication token Ngrok* dan menjalankan aplikasi Streamlit secara lokal. Token diperlukan agar *Ngrok* dapat membuat tunnel dengan akun pengguna. Setelah token diatur, aplikasi Streamlit dijalankan menggunakan subprocess sehingga server lokal aktif di background.

Apabila Anda belum mengetahui cara mendapatkan *authentication token Ngrok* milik Anda sendiri, silahkan baca pada bagian **tips and tricks submission**.

In [ ]:
import sys
import time
import socket
from dotenv import load_dotenv
import os

load_dotenv()
auth_token = os.getenv("AUTH_TOKEN")

# Start Streamlit using the same Python interpreter as this kernel
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.fileWatcherType", "none"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait until port 8501 is accepting connections (max 60 s)
print("Waiting for Streamlit to start...", end="")
for _ in range(60):
    try:
        with socket.create_connection(("localhost", 8501), timeout=1):
            break
    except OSError:
        time.sleep(1)
        print(".", end="", flush=True)
else:
    raise RuntimeError("Streamlit did not start within 60 seconds.")

print("\nStreamlit is ready!")

# Now open the ngrok tunnel
ngrok.set_auth_token(auth_token)
public_url = ngrok.connect(8501).public_url
print(f"Public URL: {public_url}")


Waiting for Streamlit to start...
Streamlit is ready!
Public URL: https://illusioned-postsystolic-sana.ngrok-free.dev


## **Membuat Public URL**
Cell ini membuat *tunnel Ngrok* ke port lokal tempat Streamlit berjalan (default: 8501). *Ngrok* kemudian menghasilkan public URL yang bisa diakses dari internet, sehingga aplikasi Streamlit dapat dibuka tanpa harus berada di jaringan lokal yang sama.

In [9]:
# Print the active tunnel URL (already opened in the cell above)
tunnels = ngrok.get_tunnels()
if tunnels:
    print(tunnels[0].public_url)
else:
    print("No active tunnels. Run the launch cell first.")


https://illusioned-postsystolic-sana.ngrok-free.dev


Apabila Anda mengalami limit endpoint usage pada Ngrok, jalankan hidden cell di bawah ini!

## **Menutup Semua Tunnel Ngrok yang Aktif**
Cell ini digunakan untuk menghentikan seluruh koneksi Ngrok yang masih aktif.

In [17]:
ngrok.kill()
print("All active ngrok tunnels have been closed.")

All active ngrok tunnels have been closed.
